# 8주차 A회차 실습 과제: 뉴스 기사 토픽 모델링

이 노트북은 `docs/ch8A.md`의 코딩 시연을 그대로 옮긴 것이 아니라, 하나의 과제를 처음부터 끝까지 수행하도록 구성한 실습용 노트북이다.

## 과제

가상의 한국어 뉴스 기사 묶음에서 BERTopic으로 숨겨진 주제를 찾고, 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 붙인다.

## 산출물

- `data/output/ch8A_topic_info.csv`: 발견된 주제 요약
- `data/output/ch8A_document_topics.csv`: 문서별 주제 배정 결과
- `data/output/ch8A_topic_keywords.json`: 주제별 핵심 단어
- `data/output/ch8A_topic_barchart.html`: 주제별 핵심 단어 시각화
- `data/output/ch8A_topic_heatmap.html`: 주제 간 유사도 시각화
- `data/output/ch8A_documents.html`: 문서 임베딩 지도
- `data/output/ch8A_topic_hierarchy.html`: 주제 계층 구조 / Topic Network
- `data/output/ch8A_term_rank.html`: 단어 순위 시각화
- `data/output/ch8A_doc_topic_distribution.html`: 특정 문서의 주제 분포
- `data/output/ch8A_topics_over_time_result.csv`: 시간별 주제 빈도
- `data/output/ch8A_topics_over_time.html`: Dynamic Topic Modeling 시각화


## 0. 환경 확인

루트에서 `setup_env.py`를 실행하면 `.venv`가 생성되고 기본 패키지가 설치된다. Jupyter에서는 `2026NLP` 커널을 선택한 뒤 이 노트북을 실행한다.

```bash
cd /Users/callii/Documents/2026-1/2026NLP
python setup_env.py
source .venv/bin/activate
python -m ipykernel install --user --name 2026nlp --display-name "2026NLP"
```


In [30]:
# ============================================================
# 루트 가상환경 및 chapter8 패키지 확인
# ============================================================
from pathlib import Path
import importlib.util
import subprocess
import sys

ROOT_DIR = Path("/Users/callii/Documents/2026-1/2026NLP")
ROOT_VENV = ROOT_DIR / ".venv"
REQUIREMENTS = ROOT_DIR / "practice" / "chapter8" / "requirements.txt"

if Path(sys.prefix).resolve() != ROOT_VENV.resolve():
    raise RuntimeError(
        "현재 노트북 커널이 루트 .venv가 아닙니다. "
        "루트에서 `python setup_env.py`를 실행한 뒤 Jupyter 커널을 `2026NLP`로 선택하세요.\n"
        f"현재 Python: {sys.executable}\n"
        f"필요한 가상환경: {ROOT_VENV}"
    )

required_modules = {
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "plotly": "plotly",
    "kiwipiepy": "kiwipiepy",
}

missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    print(f"설치되지 않은 chapter8 패키지: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)])
else:
    print("chapter8 패키지 확인 완료")

print(f"사용 중인 Python: {sys.executable}")
print(f"가상환경: {sys.prefix}")


chapter8 패키지 확인 완료
사용 중인 Python: /Users/callii/Documents/2026-1/2026NLP/.venv/bin/python
가상환경: /Users/callii/Documents/2026-1/2026NLP/.venv


In [31]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import torch
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from kiwipiepy import Kiwi
    from umap import UMAP
except ImportError as exc:
    raise ImportError(
        "필수 패키지가 설치되어 있지 않습니다. "
        "루트에서 `python -m pip install -r practice/chapter8/requirements.txt`를 실행하세요."
    ) from exc

SEED = 42
np.random.seed(SEED)
BASE_DIR = Path.cwd()
if BASE_DIR.name != "chapter8":
    BASE_DIR = Path("/Users/callii/Documents/2026-1/2026NLP/practice/chapter8")

OUTPUT_DIR = BASE_DIR / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"실행 디바이스: {device}")
print(f"산출물 저장 위치: {OUTPUT_DIR}")


PyTorch: 2.11.0
실행 디바이스: mps
산출물 저장 위치: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output


## 1. 과제 데이터 로드

`data/input/ch8A_topics_over_time_input.csv`에 미리 준비된 시계열 뉴스 데이터를 불러온다. 이 데이터는 처음부터 Dynamic Topic Modeling을 염두에 두고 `date`, `text`, `true_topic` 컬럼을 포함한다. `date`는 시간별 주제 변화 분석에 사용한다. 한국어 키워드 품질을 높이기 위해 Kiwi로 명사만 추출한 `text_nouns`를 topic representation에 사용하고, `true_topic`은 모델 학습에는 사용하지 않으며 마지막 검증 단계에서만 비교한다.


In [32]:
INPUT_PATH = BASE_DIR / "data" / "input" / "ch8A_topics_over_time_input.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"과제 데이터 파일을 찾을 수 없습니다: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)
required_columns = {"date", "text", "true_topic"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"데이터에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["date", "text"]).sort_values("date").reset_index(drop=True)
docs_raw = df["text"].astype(str).tolist()
timestamps = df["date"].tolist()

kiwi = Kiwi()
noun_tags = {"NNG", "NNP", "SL"}
stop_nouns = {"뉴스", "기자", "이번", "관련", "당국", "전문가"}

def extract_nouns(text):
    nouns = []
    for token in kiwi.tokenize(text):
        if token.tag in noun_tags and len(token.form) >= 2 and token.form not in stop_nouns:
            nouns.append(token.form)
    return " ".join(nouns)

df["text_nouns"] = [extract_nouns(text) for text in docs_raw]
docs = df["text_nouns"].tolist()

empty_docs = df[df["text_nouns"].str.len() == 0]
if len(empty_docs) > 0:
    raise ValueError(f"명사 추출 결과가 비어 있는 문서가 있습니다: {empty_docs.index.tolist()}")

print(f"데이터 파일: {INPUT_PATH}")
print(f"문서 수: {len(docs_raw)}")
print(f"기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print("주제별 문서 수:")
print(df["true_topic"].value_counts().sort_index())
display(df[["date", "text", "text_nouns", "true_topic"]].head(8))


데이터 파일: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/input/ch8A_topics_over_time_input.csv
문서 수: 60
기간: 2024-01-05 ~ 2025-03-12
주제별 문서 수:
true_topic
건강     12
경제     12
기술     12
스포츠    12
정치     12
Name: count, dtype: int64


,date,text,text_nouns,true_topic
0,2024-01-05,금리 인상과 환율 변동으로 국내 증시가 큰 폭으로 출렁였다. 반도체 수출 회복 기대...,금리 인상 환율 변동 국내 증시 반도체 수출 회복 기대감 기업 실적 전망 개선,경제
1,2024-01-12,보건 당국은 독감 환자 증가에 따라 고위험군 백신 접종을 권고했다. 대학병원 연구팀...,보건 독감 환자 증가 고위험군 백신 접종 권고 대학 병원 연구 항암 치료제 임상 결...,건강
2,2024-01-18,반도체 수출 회복 기대감에 기업 실적 전망이 개선됐다. 소비자 물가 상승률이 둔화되...,반도체 수출 회복 기대감 기업 실적 전망 개선 소비자 물가 상승 둔화 기준 금리 동...,경제
3,2024-02-02,소비자 물가 상승률이 둔화되며 기준금리 동결 가능성이 커졌다. 대기업의 설비 투자가...,소비자 물가 상승 둔화 기준 금리 동결 가능 기업 설비 투자 확대 고용 지표 개선,경제
4,2024-02-08,국회는 민생 법안을 두고 여야 협상을 이어갔지만 합의에 이르지 못했다. 정부는 지방...,국회 민생 법안 여야 협상 합의 정부 지방 재정 지원 확대 행정 개편 방안 발표,정치
5,2024-02-15,대학병원 연구팀은 새로운 항암 치료제의 임상 결과를 공개했다. 응급실 과밀 문제를 ...,대학 병원 연구 항암 치료제 임상 결과 공개 응급실 과밀 문제 지역 의료 전달 체계 개편,건강
6,2024-02-20,대기업의 설비 투자가 확대되면서 고용 지표가 개선됐다. 국제 유가 하락으로 항공과 ...,기업 설비 투자 확대 고용 지표 개선 국제 유가 하락 항공 물류 업종 비용 부담,경제
7,2024-03-01,정부는 지방 재정 지원 확대와 행정 개편 방안을 발표했다. 대통령은 국무회의에서 규...,정부 지방 재정 지원 확대 행정 개편 방안 발표 대통령 국무 회의 규제 완화 산업 ...,정치


## 2. BERTopic 모델 구성과 학습

파이프라인은 `원문 문서 임베딩 → UMAP 차원 축소 → HDBSCAN 클러스터링 → 명사 기반 c-TF-IDF 주제 표현` 순서로 실행된다.


In [33]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

umap_model = UMAP(
    n_neighbors=5,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=2,
    ngram_range=(1, 1),
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    nr_topics=5,
    verbose=False,
)

embeddings = embedding_model.encode(docs_raw, show_progress_bar=True)
topics, probabilities = topic_model.fit_transform(docs, embeddings=embeddings)
df["topic"] = topics

topic_info = topic_model.get_topic_info()
display(topic_info)

n_topics = len([t for t in set(topics) if t != -1])
n_noise = sum(1 for t in topics if t == -1)
print(f"발견된 주제 수: {n_topics}")
print(f"노이즈 문서 수: {n_noise}")


Batches: 100%|██████████| 2/2 [00:00<00:00,  8.08it/s]


,Topic,Count,Name,Representation,Representative_Docs
0,-1,4,-1_확대_기업_개선_비용,"[확대, 기업, 개선, 비용, 국제, 물류, 부담, 업종, 유가, 하락]",[소비자 물가 상승 둔화 기준 금리 동결 가능 기업 설비 투자 확대 고용 지표 개선...
1,0,24,0_국내_발표_개편_개선,"[국내, 발표, 개편, 개선, 절차, 기준, 강화, 기업, , ]",[국내 연구진 거대 언어 모델 추론 속도 개선 기법 발표 클라우드 기업 데이터 센터...
2,1,12,1_확대_개편_발표_강화,"[확대, 개편, 발표, 강화, 절차, , , , , ]",[정부 지방 재정 지원 확대 행정 개편 방안 발표 대통령 국무 회의 규제 완화 산업...
3,2,12,2____,"[, , , , , , , , , ]",[감독 선수 수비 집중력 승부 평가 리그 선두 원정 경기 승리 플레이오프 진출 확정...
4,3,8,3_금리_기업_국내_개선,"[금리, 기업, 국내, 개선, 소비자, 상승, 물가, 가능, 둔화, 동결]",[반도체 수출 회복 기대감 기업 실적 전망 개선 소비자 물가 상승 둔화 기준 금리 ...


발견된 주제 수: 4
노이즈 문서 수: 4


## 3. 주제별 핵심 단어와 대표 문서 확인

아래 결과를 보고 각 주제가 무엇을 의미하는지 직접 이름을 붙인다.


In [34]:
representative_docs_by_topic = topic_model.get_representative_docs()
topic_keywords = {}
topic_keyword_scores = {}
valid_topic_ids = sorted(t for t in df["topic"].unique() if t != -1)
grouped_texts = [" ".join(df.loc[df["topic"] == topic_id, "text_nouns"]) for topic_id in valid_topic_ids]

keyword_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=1,
    ngram_range=(1, 1),
    max_features=500,
)
topic_term_matrix = keyword_vectorizer.fit_transform(grouped_texts)
terms = keyword_vectorizer.get_feature_names_out()

for row_idx, topic_id in enumerate(valid_topic_ids):
    scores = topic_term_matrix[row_idx].toarray().ravel()
    top_indices = scores.argsort()[::-1][:8]

    keyword_pairs = [(terms[i], float(scores[i])) for i in top_indices if scores[i] > 0]
    keywords = [word for word, score in keyword_pairs]
    topic_keywords[int(topic_id)] = keywords
    topic_keyword_scores[int(topic_id)] = keyword_pairs

    representative_docs = representative_docs_by_topic.get(topic_id, [])[:2]
    print("=" * 80)
    print(f"주제 {topic_id}")
    print(f"핵심 단어: {', '.join(keywords)}")
    print("대표 문서:")
    for doc in representative_docs:
        print(f"- {doc}")

with open(OUTPUT_DIR / "ch8A_topic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)


주제 0
핵심 단어: 주행, 병원, 항암, 대학, 추론, 공개, 치료제, 속도
대표 문서:
- 국내 연구진 거대 언어 모델 추론 속도 개선 기법 발표 클라우드 기업 데이터 센터 전력 효율 개발
- 국내 연구진 거대 언어 모델 추론 속도 개선 기법 발표 클라우드 기업 데이터 센터 전력 효율 개발
주제 1
핵심 단어: 여야, 협상, 확대, 방안, 재정, 지원, 행정, 정부
대표 문서:
- 정부 지방 재정 지원 확대 행정 개편 방안 발표 대통령 국무 회의 규제 완화 산업 경쟁력 강화 주문
- 정부 지방 재정 지원 확대 행정 개편 방안 발표 대통령 국무 회의 규제 완화 산업 경쟁력 강화 주문
주제 2
핵심 단어: 승리, 추가, 대표, 축구, 시간, 후반, 결승, 프로
대표 문서:
- 감독 선수 수비 집중력 승부 평가 리그 선두 원정 경기 승리 플레이오프 진출 확정
- 프로 야구 선발 투수 실점 호투 연승 축구 대표 후반 추가 시간 결승 승리
주제 3
핵심 단어: 금리, 전망, 반도체, 수출, 회복, 기대감, 실적, 인상
대표 문서:
- 반도체 수출 회복 기대감 기업 실적 전망 개선 소비자 물가 상승 둔화 기준 금리 동결 가능
- 반도체 수출 회복 기대감 기업 실적 전망 개선 소비자 물가 상승 둔화 기준 금리 동결 가능


## 4. 실제 주제와 모델 주제 비교

실제 수업에서는 정답 레이블이 없는 경우가 많다. 여기서는 실습 검증을 위해 생성 시 사용한 `true_topic`과 BERTopic의 `topic`을 비교한다.


In [35]:
comparison = pd.crosstab(df["topic"], df["true_topic"], margins=True)
display(comparison)

topic_label_guess = {}
for topic_id in sorted(t for t in df["topic"].unique() if t != -1):
    topic_docs = df[df["topic"] == topic_id]
    majority_label = topic_docs["true_topic"].value_counts().idxmax()
    purity = topic_docs["true_topic"].value_counts(normalize=True).max()
    topic_label_guess[int(topic_id)] = majority_label
    print(f"주제 {topic_id}: 추정 이름 = {majority_label}, 순도 = {purity:.2f}, 문서 수 = {len(topic_docs)}")


true_topic,건강,경제,기술,스포츠,정치,All
topic,,,,,,
-1,0,4,0,0,0,4
0,12,0,12,0,0,24
1,0,0,0,0,12,12
2,0,0,0,12,0,12
3,0,8,0,0,0,8
All,12,12,12,12,12,60


주제 0: 추정 이름 = 건강, 순도 = 0.50, 문서 수 = 24
주제 1: 추정 이름 = 정치, 순도 = 1.00, 문서 수 = 12
주제 2: 추정 이름 = 스포츠, 순도 = 1.00, 문서 수 = 12
주제 3: 추정 이름 = 경제, 순도 = 1.00, 문서 수 = 8


## 5. 특정 문서의 주제 분석

문서 하나를 골라 모델이 어떤 주제로 분류했는지 확인하고, 분류가 타당한지 해석한다.


In [36]:
doc_idx = 0
doc_topic = int(df.loc[doc_idx, "topic"])
doc_true_topic = df.loc[doc_idx, "true_topic"]
doc_text = df.loc[doc_idx, "text"]
doc_nouns = df.loc[doc_idx, "text_nouns"]

print(f"문서 번호: {doc_idx}")
print(f"문서 날짜: {df.loc[doc_idx, 'date'].date()}")
print(f"문서 내용: {doc_text}")
print(f"명사 추출: {doc_nouns}")
print(f"실제 주제: {doc_true_topic}")
print(f"모델 주제: {doc_topic}")
print(f"모델 주제 이름 추정: {topic_label_guess.get(doc_topic, '노이즈')}")

doc_distribution_df = pd.DataFrame()
if probabilities is not None:
    probs = np.asarray(probabilities)
    if probs.ndim == 2 and doc_idx < len(probs):
        probability_values = probs[doc_idx]
        probability_topics = valid_topic_ids[: len(probability_values)]
        doc_distribution_df = pd.DataFrame({
            "topic": probability_topics,
            "estimated_label": [topic_label_guess.get(int(t), f"주제 {t}") for t in probability_topics],
            "probability": probability_values,
        }).sort_values("probability", ascending=False)
        display(doc_distribution_df)

        import plotly.express as px
        fig = px.bar(
            doc_distribution_df,
            x="estimated_label",
            y="probability",
            text="probability",
            title=f"문서 {doc_idx}의 주제 분포",
        )
        fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
        fig.update_layout(xaxis_title="주제", yaxis_title="확률", yaxis_range=[0, 1.05])
        output_path = OUTPUT_DIR / "ch8A_doc_topic_distribution.html"
        fig.write_html(output_path)
        print(f"저장 완료: {output_path}")
    elif probs.ndim == 1 and doc_idx < len(probs):
        print(f"\n배정 신뢰도: {probs[doc_idx]:.3f}")


문서 번호: 0
문서 날짜: 2024-01-05
문서 내용: 금리 인상과 환율 변동으로 국내 증시가 큰 폭으로 출렁였다. 반도체 수출 회복 기대감에 기업 실적 전망이 개선됐다.
명사 추출: 금리 인상 환율 변동 국내 증시 반도체 수출 회복 기대감 기업 실적 전망 개선
실제 주제: 경제
모델 주제: 3
모델 주제 이름 추정: 경제


,topic,estimated_label,probability
3,3,경제,1.000000e+00
0,0,건강,3.584968e-309
2,2,스포츠,2.446520e-309
1,1,정치,9.545680e-310


저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_doc_topic_distribution.html


## 6. 시각화 저장

Plotly 기반 HTML 파일은 브라우저에서 열어 상호작용하며 확인할 수 있다.


In [37]:
visualization_outputs = []

try:
    import plotly.express as px

    keyword_rows = []
    for topic_id, pairs in topic_keyword_scores.items():
        label = topic_label_guess.get(topic_id, f"주제 {topic_id}")
        for word, score in pairs:
            keyword_rows.append({"topic": f"주제 {topic_id}: {label}", "word": word, "score": score})

    keyword_df = pd.DataFrame(keyword_rows)
    fig = px.bar(
        keyword_df,
        x="score",
        y="word",
        color="topic",
        facet_col="topic",
        facet_col_wrap=2,
        orientation="h",
        height=900,
        title="주제별 핵심 단어 TF-IDF 점수",
    )
    fig.update_yaxes(matches=None, autorange="reversed")
    fig.update_xaxes(matches=None)
    fig.update_layout(showlegend=False)
    output_path = OUTPUT_DIR / "ch8A_topic_barchart.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"barchart 저장 실패: {exc}")

try:
    fig = topic_model.visualize_heatmap(top_n_topics=min(8, n_topics))
    output_path = OUTPUT_DIR / "ch8A_topic_heatmap.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"heatmap 저장 실패: {exc}")


try:
    fig = topic_model.visualize_hierarchy()
    output_path = OUTPUT_DIR / "ch8A_topic_hierarchy.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"BERTopic hierarchy 생성 실패, 주제 유사도 네트워크로 대체합니다: {exc}")
    import plotly.graph_objects as go
    from sklearn.metrics.pairwise import cosine_similarity

    network_topic_ids = valid_topic_ids
    embeddings_for_network = embeddings
    centroids = np.vstack([
        embeddings_for_network[df["topic"].to_numpy() == topic_id].mean(axis=0)
        for topic_id in network_topic_ids
    ])
    similarity = cosine_similarity(centroids)

    angles = np.linspace(0, 2 * np.pi, len(network_topic_ids), endpoint=False)
    x = np.cos(angles)
    y = np.sin(angles)
    edge_x, edge_y = [], []
    for i in range(len(network_topic_ids)):
        for j in range(i + 1, len(network_topic_ids)):
            if similarity[i, j] >= 0.35:
                edge_x += [x[i], x[j], None]
                edge_y += [y[i], y[j], None]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="#999"), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        text=[f"주제 {topic_id}<br>{topic_label_guess.get(int(topic_id), '')}" for topic_id in network_topic_ids],
        textposition="top center",
        marker=dict(size=28, color=list(range(len(network_topic_ids))), colorscale="Viridis", showscale=False),
        hovertemplate="%{text}<extra></extra>",
    ))
    fig.update_layout(
        title="Topic Network: 주제 중심점 간 유사도",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        showlegend=False,
        height=650,
    )
    output_path = OUTPUT_DIR / "ch8A_topic_hierarchy.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)

try:
    fig = topic_model.visualize_term_rank()
    output_path = OUTPUT_DIR / "ch8A_term_rank.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"term rank 저장 실패: {exc}")

try:
    fig = topic_model.visualize_documents(docs_raw, embeddings=embeddings)
    output_path = OUTPUT_DIR / "ch8A_documents.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"documents 저장 실패: {exc}")

for path in visualization_outputs:
    print(f"저장 완료: {path}")


저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topic_barchart.html
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topic_heatmap.html
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topic_hierarchy.html
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_term_rank.html
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_documents.html


## 7. Dynamic Topic Modeling

`date` 컬럼을 사용해 시간에 따라 각 주제의 빈도가 어떻게 변하는지 확인한다.


In [38]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    topics=topics,
    nr_bins=6,
)

topics_over_time_path = OUTPUT_DIR / "ch8A_topics_over_time_result.csv"
topics_over_time.to_csv(topics_over_time_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {topics_over_time_path}")
display(topics_over_time.head(12))

try:
    fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=min(5, n_topics))
    output_path = OUTPUT_DIR / "ch8A_topics_over_time.html"
    fig.write_html(output_path)
    print(f"저장 완료: {output_path}")
except Exception as exc:
    print(f"dynamic topic 시각화 저장 실패: {exc}")


저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topics_over_time_result.csv


,Topic,Words,Frequency,Timestamp
0,-1,"확대, 기업, 개선, 업종, 항공",2,2024-01-04 13:37:55.200
1,0,"개편, 국내, 발표, 개선, 절차",2,2024-01-04 13:37:55.200
2,1,"확대, 개편, 발표, 강화, 절차",2,2024-01-04 13:37:55.200
3,2,", , , ,",1,2024-01-04 13:37:55.200
4,3,"금리, 국내, 기업, 개선, 소비자",3,2024-01-04 13:37:55.200
5,-1,"소비자, 가능, 동결, 둔화, 물가",1,2024-03-17 00:00:00.000
6,0,"국내, 발표, 개선, 기업, 개편",2,2024-03-17 00:00:00.000
7,1,"확대, 강화, 개편, 절차, 발표",5,2024-03-17 00:00:00.000
8,2,", , , ,",2,2024-03-17 00:00:00.000
9,3,"금리, 기업, 개선, 소비자, 상승",2,2024-03-17 00:00:00.000


저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topics_over_time.html


## 8. 결과 저장과 제출용 질문

아래 파일을 저장한 뒤, 다음 질문에 답한다.

1. BERTopic이 발견한 주제 수는 몇 개인가?
2. 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 어떻게 붙일 수 있는가?
3. 노이즈 문서는 몇 개이며, 노이즈가 생긴 이유는 무엇이라고 해석할 수 있는가?
4. 실제 주제와 모델 주제가 잘 맞지 않는 사례가 있다면 왜 그런가?


In [39]:
topic_info_path = OUTPUT_DIR / "ch8A_topic_info.csv"
document_topics_path = OUTPUT_DIR / "ch8A_document_topics.csv"

topic_info.to_csv(topic_info_path, index=False, encoding="utf-8-sig")
df.to_csv(document_topics_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {topic_info_path}")
print(f"저장 완료: {document_topics_path}")
print(f"저장 완료: {OUTPUT_DIR / 'ch8A_topic_keywords.json'}")

display(df.head(10))


저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topic_info.csv
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_document_topics.csv
저장 완료: /Users/callii/Documents/2026-1/2026NLP/practice/chapter8/data/output/ch8A_topic_keywords.json


,date,text,true_topic,text_nouns,topic
0,2024-01-05,금리 인상과 환율 변동으로 국내 증시가 큰 폭으로 출렁였다. 반도체 수출 회복 기대...,경제,금리 인상 환율 변동 국내 증시 반도체 수출 회복 기대감 기업 실적 전망 개선,3
1,2024-01-12,보건 당국은 독감 환자 증가에 따라 고위험군 백신 접종을 권고했다. 대학병원 연구팀...,건강,보건 독감 환자 증가 고위험군 백신 접종 권고 대학 병원 연구 항암 치료제 임상 결...,0
2,2024-01-18,반도체 수출 회복 기대감에 기업 실적 전망이 개선됐다. 소비자 물가 상승률이 둔화되...,경제,반도체 수출 회복 기대감 기업 실적 전망 개선 소비자 물가 상승 둔화 기준 금리 동...,3
3,2024-02-02,소비자 물가 상승률이 둔화되며 기준금리 동결 가능성이 커졌다. 대기업의 설비 투자가...,경제,소비자 물가 상승 둔화 기준 금리 동결 가능 기업 설비 투자 확대 고용 지표 개선,-1
4,2024-02-08,국회는 민생 법안을 두고 여야 협상을 이어갔지만 합의에 이르지 못했다. 정부는 지방...,정치,국회 민생 법안 여야 협상 합의 정부 지방 재정 지원 확대 행정 개편 방안 발표,1
5,2024-02-15,대학병원 연구팀은 새로운 항암 치료제의 임상 결과를 공개했다. 응급실 과밀 문제를 ...,건강,대학 병원 연구 항암 치료제 임상 결과 공개 응급실 과밀 문제 지역 의료 전달 체계 개편,0
6,2024-02-20,대기업의 설비 투자가 확대되면서 고용 지표가 개선됐다. 국제 유가 하락으로 항공과 ...,경제,기업 설비 투자 확대 고용 지표 개선 국제 유가 하락 항공 물류 업종 비용 부담,-1
7,2024-03-01,정부는 지방 재정 지원 확대와 행정 개편 방안을 발표했다. 대통령은 국무회의에서 규...,정치,정부 지방 재정 지원 확대 행정 개편 방안 발표 대통령 국무 회의 규제 완화 산업 ...,1
8,2024-03-04,국제 유가 하락으로 항공과 물류 업종의 비용 부담이 줄었다. 금리 인상과 환율 변동...,경제,국제 유가 하락 항공 물류 업종 비용 부담 금리 인상 환율 변동 국내 증시,3
9,2024-03-10,프로야구 선발 투수가 무실점 호투로 팀의 연승을 이끌었다. 축구 대표팀은 후반 추가...,스포츠,프로 야구 선발 투수 실점 호투 연승 축구 대표 후반 추가 시간 결승 승리,2


---

# 8주차 B회차: BERTopic 다듬기 + 동적/멀티모달 확장

여기서부터는 `docs/ch8B.md`와 짝이 되는 셀이다. 위(0~8절)에서 만든 `topic_model`, `documents`(=`docs`), `topics`, `probabilities`, `embeddings`를 그대로 이어 사용한다.


## 9. 토픽 평가지표 — NPMI · Topic Diversity

지표 한 줄 정리: **일관성**(NPMI, 높을수록 좋음, 범위 −1~1)과 **다양성**(Topic Diversity, 0~1, 높을수록 좋음)을 함께 본다. 본격적인 C_v는 외부 라이브러리(`gensim`)가 필요하므로, 여기서는 의존을 최소화한 **자체 구현** NPMI + Topic Diversity로 직관을 본다.


In [ ]:
from collections import Counter
from itertools import combinations


def topic_diversity(topic_model, top_n: int = 10) -> float:
    """모든 토픽의 상위 N개 단어 중 고유 단어 비율 (0~1, ↑ 좋음)."""
    topics_ = [t for t in topic_model.get_topics().keys() if t != -1]
    all_words: list[str] = []
    for t in topics_:
        words = [w for w, _ in topic_model.get_topic(t)[:top_n]]
        all_words.extend(words)
    return len(set(all_words)) / max(len(all_words), 1)


def npmi_coherence(topic_model, docs_for_eval, top_n: int = 10) -> float:
    """문서 동시 출현 기반의 NPMI 평균 (간이 구현, ↑ 좋음, 범위 −1~1)."""
    tokenized = [set(d.split()) for d in docs_for_eval]
    N = len(tokenized)
    df_count = Counter(w for d in tokenized for w in d)
    scores: list[float] = []
    topics_ = [t for t in topic_model.get_topics().keys() if t != -1]
    for t in topics_:
        words = [w for w, _ in topic_model.get_topic(t)[:top_n]]
        for w_i, w_j in combinations(words, 2):
            n_ij = sum(1 for d in tokenized if w_i in d and w_j in d)
            if n_ij == 0 or df_count[w_i] == 0 or df_count[w_j] == 0:
                continue
            p_ij = n_ij / N
            p_i = df_count[w_i] / N
            p_j = df_count[w_j] / N
            pmi = np.log(p_ij / (p_i * p_j))
            npmi = pmi / (-np.log(p_ij))
            scores.append(npmi)
    return float(np.mean(scores)) if scores else 0.0


eval_npmi = npmi_coherence(topic_model, docs)
eval_div = topic_diversity(topic_model)
print(f"NPMI            : {eval_npmi:+.4f}  (높을수록 좋음, −1~1)")
print(f"Topic Diversity : {eval_div:.4f}  (높을수록 좋음, 0~1)")

eval_summary = pd.DataFrame(
    [{"metric": "NPMI", "value": eval_npmi, "criteria": ">0.05 수용, >0.15 양호"},
     {"metric": "Topic Diversity", "value": eval_div, "criteria": ">0.7 양호"}]
)
display(eval_summary)
eval_summary.to_csv(OUTPUT_DIR / "ch8B_eval_metrics.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_eval_metrics.csv'}")


## 10. 노이즈 처리 — `reduce_outliers()`

HDBSCAN이 -1로 분리한 문서를 **가장 유사한 토픽**에 다시 배정한다. 노이즈가 많을 때 유효하다. 이 데이터셋은 `nr_topics=5`로 명시 지정하여 학습되었으므로 노이즈가 적을 수 있다 — 그런 경우에는 셀이 자동으로 알려주고 안전하게 통과한다.


In [ ]:
topics_array = np.asarray(topics)
n_outliers_before = int((topics_array == -1).sum())
print(f"노이즈(-1) 문서 수: {n_outliers_before} / {len(topics_array)}")

if n_outliers_before == 0:
    print("이 모델에는 노이즈가 없어 reduce_outliers()는 변화를 만들지 않는다.")
    new_topics_list = list(topics_array)
else:
    new_topics_list = topic_model.reduce_outliers(
        documents=docs,
        topics=list(topics_array),
        strategy="distributions",
    )
    n_outliers_after = sum(1 for t in new_topics_list if t == -1)
    print(f"재할당 후 남은 노이즈: {n_outliers_after}")

# 전후 비교 표
before_counts = pd.Series(topics_array).value_counts().rename("before").sort_index()
after_counts = pd.Series(new_topics_list).value_counts().rename("after").sort_index()
compare_df = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)
compare_df["delta"] = compare_df["after"] - compare_df["before"]
display(compare_df)
compare_df.to_csv(OUTPUT_DIR / "ch8B_outlier_reduction.csv", encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_outlier_reduction.csv'}")


## 11. 토픽 축소·병합 — `reduce_topics()`

비슷한 토픽을 c-TF-IDF 코사인 유사도 기준으로 자동 병합한다. 이 데이터에서는 5개 토픽을 4개로 줄여 본다. 병합 후 키워드는 자동 재계산된다.

> **주의**: `reduce_topics()`는 토픽 모델의 내부 상태를 영구적으로 변경한다. 다음 셀에서는 이 변화를 반영한 `topic_model`이 사용된다.


In [ ]:
before_info = topic_model.get_topic_info().copy()
n_before = len([t for t in before_info["Topic"].tolist() if t != -1])

target_nr = max(2, n_before - 1)  # 데이터가 적으므로 1개만 줄여 본다
print(f"원래 토픽 수: {n_before}  →  목표 토픽 수: {target_nr}")

topic_model.reduce_topics(docs, nr_topics=target_nr)

after_info = topic_model.get_topic_info()
n_after = len([t for t in after_info["Topic"].tolist() if t != -1])
print(f"축소 후 토픽 수: {n_after}")

display(after_info)

reduced_keywords = {
    int(t): [w for w, _ in topic_model.get_topic(t)[:10]]
    for t in topic_model.get_topics().keys() if t != -1
}
with open(OUTPUT_DIR / "ch8B_reduced_topic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(reduced_keywords, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_reduced_topic_keywords.json'}")

# 축소 후 평가지표 재계산
post_npmi = npmi_coherence(topic_model, docs)
post_div = topic_diversity(topic_model)
print(f"\n축소 후 NPMI            : {post_npmi:+.4f}  (이전 {eval_npmi:+.4f})")
print(f"축소 후 Topic Diversity : {post_div:.4f}  (이전 {eval_div:.4f})")


## 12. Optuna 다목적 최적화 — 일관성↑ × 노이즈↓

UMAP·HDBSCAN의 파라미터를 베이지안 최적화로 자동 탐색한다. 두 목표(일관성 최대화, 노이즈 비율 최소화)는 서로 충돌하므로 결과는 단일 최적 해가 아니라 **파레토 프론트** — trade-off의 후보들 — 가 나온다.

> **강의 데모**: `n_trials=10`. 실무에서는 50~100회. 이 데이터셋이 작아 일부 trial은 1개 토픽만 만드는 식으로 실패할 수 있고, 그런 trial은 NPMI=0으로 떨어져 자동으로 후순위가 된다.


In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 15)
    n_components = trial.suggest_int("n_components", 2, 8)
    min_cluster_size = trial.suggest_int("min_cluster_size", 3, 12)
    min_samples = trial.suggest_int("min_samples", 1, 5)

    umap_t = UMAP(
        n_neighbors=n_neighbors, n_components=n_components,
        min_dist=0.0, metric="cosine", random_state=SEED,
    )
    hdbscan_t = HDBSCAN(
        min_cluster_size=min_cluster_size, min_samples=min_samples,
        metric="euclidean", cluster_selection_method="eom",
    )
    vectorizer_t = CountVectorizer(
        token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}", min_df=1, ngram_range=(1, 1),
    )
    model_t = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_t, hdbscan_model=hdbscan_t,
        vectorizer_model=vectorizer_t, language="multilingual",
        calculate_probabilities=False, verbose=False,
    )
    try:
        topics_t, _ = model_t.fit_transform(docs, embeddings=embeddings)
    except Exception:
        return 0.0, 1.0

    n_t = len([t for t in set(topics_t) if t != -1])
    if n_t < 2:
        return 0.0, 1.0  # 토픽이 1개면 의미 있는 평가 불가

    coh = npmi_coherence(model_t, docs)
    noise_ratio = sum(1 for t in topics_t if t == -1) / len(topics_t)
    return coh, noise_ratio


study = optuna.create_study(directions=["maximize", "minimize"])
study.optimize(objective, n_trials=10, show_progress_bar=False)

# 파레토 프론트 출력
pareto_rows = []
for t in study.best_trials:
    pareto_rows.append({
        "trial": t.number,
        "NPMI": round(t.values[0], 4),
        "noise_ratio": round(t.values[1], 4),
        **t.params,
    })
pareto_df = pd.DataFrame(pareto_rows).sort_values(["NPMI", "noise_ratio"], ascending=[False, True])
display(pareto_df)
pareto_df.to_csv(OUTPUT_DIR / "ch8B_optuna_pareto.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_optuna_pareto.csv'}")

print("\n[해석 가이드]")
print("- NPMI가 높고 노이즈 비율도 낮은 trial이 이상적이지만, 보통은 trade-off가 존재한다.")
print("- 자신의 분석 목적(일관성 우선 vs 데이터 손실 최소화)에 맞춰 한 trial을 선택한다.")


## 13. 토픽 간 시계열 상관 — 함께 움직이는 주제

`topics_over_time` 결과(7절에서 만든 `topics_over_time` 변수)를 피벗해 토픽×시간 행렬을 만들고 상관 행렬을 계산한다. 절대값이 큰 쌍은 함께 움직이는 주제다.


In [ ]:
# 11절 reduce_topics로 토픽 구성이 바뀌었으므로 동적 결과를 다시 만든다
topics_over_time2 = topic_model.topics_over_time(
    docs,
    timestamps,
    nr_bins=6,
)

pivot_df = topics_over_time2.pivot(
    index="Timestamp", columns="Topic", values="Frequency",
).fillna(0)
correlation_matrix = pivot_df.corr().round(3)
display(correlation_matrix)
correlation_matrix.to_csv(OUTPUT_DIR / "ch8B_topic_correlation.csv", encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_topic_correlation.csv'}")

# 절대값 0.5 이상의 토픽 쌍
strong_pairs = []
cols = correlation_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = correlation_matrix.iloc[i, j]
        if abs(v) >= 0.5:
            strong_pairs.append({"topic_a": int(cols[i]), "topic_b": int(cols[j]), "corr": float(v)})

if strong_pairs:
    print("\n강한 동행/역행 토픽 쌍 (|corr| ≥ 0.5):")
    display(pd.DataFrame(strong_pairs).sort_values("corr", key=lambda s: s.abs(), ascending=False))
else:
    print("\n|corr| ≥ 0.5인 토픽 쌍이 없다. 데이터 양/기간을 늘리면 신호가 또렷해진다.")


## 14. 토픽 생명 주기 — 출현 → 최고점 → 소멸

각 토픽이 **언제 처음 등장했고**(출현), **언제 가장 활발했고**(최고점), **언제 사라졌는지**(소멸)를 자동으로 식별한다.


In [ ]:
def analyze_topic_lifecycle(tot_df, model, threshold=0.01):
    """토픽의 출현·최고점·소멸 시점 자동 식별."""
    lifecycle = {}
    for topic in [t for t in tot_df["Topic"].unique() if t != -1]:
        td = tot_df[tot_df["Topic"] == topic].sort_values("Timestamp")
        if td.empty:
            continue
        # 출현: 빈도가 임계값을 넘는 첫 시점
        above = td[td["Frequency"] > threshold]
        emergence = above.iloc[0]["Timestamp"] if len(above) > 0 else None

        # 최고점
        peak_idx = td["Frequency"].idxmax()
        peak_t = td.loc[peak_idx, "Timestamp"]
        peak_f = float(td.loc[peak_idx, "Frequency"])

        # 소멸: 최고점 이후 임계값 밑으로 떨어진 첫 시점
        after = td.loc[peak_idx:]
        below = after[after["Frequency"] < threshold]
        decline = below.iloc[0]["Timestamp"] if len(below) > 0 else None

        lifecycle[int(topic)] = {
            "emergence": str(emergence) if emergence is not None else None,
            "peak": str(peak_t),
            "peak_frequency": peak_f,
            "decline": str(decline) if decline is not None else None,
            "keywords": [w for w, _ in model.get_topic(topic)[:5]],
        }
    return lifecycle


lifecycle = analyze_topic_lifecycle(topics_over_time2, topic_model, threshold=0.5)
lifecycle_rows = []
for topic_id, info in lifecycle.items():
    lifecycle_rows.append({
        "topic": topic_id,
        "keywords": ", ".join(info["keywords"]),
        "emergence": info["emergence"],
        "peak": info["peak"],
        "peak_frequency": info["peak_frequency"],
        "decline": info["decline"],
    })
lifecycle_df = pd.DataFrame(lifecycle_rows)
display(lifecycle_df)
lifecycle_df.to_csv(OUTPUT_DIR / "ch8B_topic_lifecycle.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_DIR / 'ch8B_topic_lifecycle.csv'}")

print("\n[해석 가이드]")
print("- emergence가 분석 시작 직후이고 decline이 비어 있으면: 안정적인 상시 토픽")
print("- emergence가 늦고 peak가 emergence 직후면: 급상승 패턴")
print("- emergence/peak는 있는데 decline이 비어 있으면: 점진 상승 중")


## 15. (선택) 멀티모달 토픽 모델링 — CLIP

> **선택 셀**입니다. 다음 모두가 갖춰진 경우에만 실행된다:
> - `pip install "bertopic[vision]" pillow` 가 끝나 있음
> - 텍스트–이미지 쌍 데이터를 별도로 준비
>
> 환경/데이터가 없으면 셀이 자동으로 안내 메시지만 출력하고 통과한다. 무거운 모델(약 600MB)을 받지 않으려면 그대로 두면 된다.


In [ ]:
# === 멀티모달 시연 (옵션) ===
RUN_MULTIMODAL = False  # 실행하려면 True로 바꾸고, 아래 데이터를 채운 뒤 다시 실행

multimodal_status = "skipped"
multimodal_reason = ""

if not RUN_MULTIMODAL:
    multimodal_reason = "RUN_MULTIMODAL = False (기본값). 실행하려면 위 변수를 True로 바꾸세요."
else:
    try:
        from bertopic.backend import MultiModalBackend
        from bertopic.representation import VisualRepresentation
    except ImportError as exc:
        multimodal_reason = f"의존성 누락: {exc}. `pip install \"bertopic[vision]\" pillow` 후 재시도."
    else:
        # === 사용자가 채워야 하는 부분 ===
        # 텍스트 캡션과 이미지 경로 리스트(같은 길이, 1:1 매칭).
        captions: list[str] = []  # 예: ["빈티지 원목 의자", "검정 가죽 백팩", ...]
        image_paths: list[str] = []  # 예: ["/path/img1.jpg", "/path/img2.jpg", ...]

        if not captions or not image_paths or len(captions) != len(image_paths):
            multimodal_reason = "captions / image_paths를 직접 채워주세요 (1:1 길이)."
        else:
            mm_emb = MultiModalBackend("clip-ViT-B-32", batch_size=16)
            mm_repr = {"Visual_Aspect": VisualRepresentation()}
            mm_model = BERTopic(
                embedding_model=mm_emb,
                representation_model=mm_repr,
                verbose=False,
            )
            mm_topics, _ = mm_model.fit_transform(documents=captions, images=image_paths)
            mm_info = mm_model.get_topic_info()
            display(mm_info)
            mm_info.to_csv(OUTPUT_DIR / "ch8B_multimodal_topic_info.csv", index=False, encoding="utf-8-sig")
            multimodal_status = "ok"
            print(f"저장 완료: {OUTPUT_DIR / 'ch8B_multimodal_topic_info.csv'}")

print(f"[multimodal] status={multimodal_status} | {multimodal_reason}")
